# 06 — Augmentation Round 2: Cutout · Grid-shift · Mixup · Channel-dropout

## Starting point from notebook 05

| Method | LODO mean | LODO CALCE | LODO NASA | GKF-6 |
|---|---|---|---|---|
| Anchor (rate_warp ±25%, jit=0.01) | 0.0645 | 0.0587 | 0.0703 | 0.0624 |
| **Optuna aug** (ws=0.07, jit=0.006, tw=0.11) | **0.0580** | 0.0514 | 0.0646 | **0.0545** |
| Optuna aug + C2 1-shot (k=1) | — | 0.0681 | **0.0452** | — |

## New augmentation axes

**Voltage-grid reframing**: the model trains on `X_base3` where the L axis is a *voltage grid*
(index 0 = V_HI, trailing = V_LO), not cycle-time.  The valid region is a *leading block*.
This means voltage amplitude warp has no operand (voltage is the axis, not a channel).
Its physical intent (CALCE 3.0 V vs NASA 2.7 V cutoff gap) maps to **end-of-discharge cutout**.

| Axis | Physical motivation | Impl |
|---|---|---|
| `cutout_frac` | CALCE/NASA cutoff-voltage gap; invariance to early discharge termination | shrink leading valid block from low-V edge |
| `gridshift_max` | OCV plateau offset (temp/chemistry); invariance to absolute V level | roll valid block ±k positions along grid |
| `mixup_alpha` | interpolated degradation states; calibration regularisation | convex combo of (x, y) pairs on shared voltage grid |
| `channel_dropout_p` | sensor robustness; prevents over-reliance on any one channel | zero one random channel per sample |

**Multi-objective sweep**: instead of collapsing CALCE+NASA MAE into a mean (which hid the
time_warp tradeoff), we optimise both simultaneously and visualise the **Pareto front** with
Plotly (hover to inspect per-trial params).

## Setup

In [ ]:
import copy, sys, pickle
from pathlib import Path
from importlib import reload

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

def find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()
    else:
        start_path = Path(start_path)
    for marker in ['.git', 'pyproject.toml', 'CLAUDE.md', '.claude']:
        for p in [start_path] + list(start_path.parents):
            if (p / marker).exists():
                return p
    raise RuntimeError('Could not find project root')

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

import src.voltage_grid as vg
import src.vg_extended as vgx
import src.vg_models
import src.vg_da
import src.vg_augment
import src.sequence
reload(src.vg_models)
reload(src.vg_da)
reload(src.vg_augment)
reload(src.sequence)

torch.manual_seed(42)
np.random.seed(42)

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')

RESULTS = ROOT / 'results' / 'extended_tier1'
RESULTS.mkdir(parents=True, exist_ok=True)
CACHE   = ROOT / 'data' / 'processed' / 'vg_extended_tier1.npz'

EPOCHS     = 300
PATIENCE   = 50
BATCH_SIZE = 64
LR         = 1e-3
SCHEDULER  = 'cosine'
N_SEEDS    = 1       # single-seed everywhere; multi-seed only for final presentation metrics
N_TRIALS   = 100     # multi-objective NSGA-II sweep (includes enqueued anchors)
USE_AMP    = DEVICE.type == 'cuda'

# nb05 Optuna best — new axes default to 0 (off)
NB05_BEST = {'warp_strength': 0.070, 'jitter_sigma': 0.006, 'time_warp_strength': 0.110}

print(f'Cache:   {CACHE}')
print(f'Results: {RESULTS}')
print(f'N_TRIALS: {N_TRIALS}  USE_AMP: {USE_AMP}')

## Load data

In [ ]:
X, mask, y, groups, cidx, ds_groups = vgx.load_npz_ext(CACHE)
X_base3 = X[..., :3]  # [C-rate, ΔT, t_elapsed]

print(f'X: {X.shape}  y: {y.shape}  cells: {len(set(groups))}')
print(f'ds_groups: {dict(zip(*np.unique(ds_groups, return_counts=True)))}')

In [ ]:
study_group_map = {
    'B0005': 'NASA_CTRL',  'B0006': 'NASA_CTRL',  'B0007': 'NASA_CTRL',  'B0018': 'NASA_CTRL',
    'RW1':   'NASA_RW',    'RW9':   'NASA_RW',    'RW13':  'NASA_RW',    'RW14':  'NASA_RW',
    'RW15':  'NASA_RW',    'RW16':  'NASA_RW',    'RW17':  'NASA_RW',    'RW19':  'NASA_RW',
    'RW20':  'NASA_RW',
    'CS2_8':  'CALCE_CS2_T1', 'CS2_21': 'CALCE_CS2_T1',
    'CS2_33': 'CALCE_CS2_T1', 'CS2_34': 'CALCE_CS2_T1',
    'CS2_35': 'CALCE_CS2_T2', 'CS2_36': 'CALCE_CS2_T2',
    'CS2_37': 'CALCE_CS2_T2', 'CS2_38': 'CALCE_CS2_T2',
    'CS2_3':  'CALCE_CS2_T3', 'CS2_9':  'CALCE_CS2_T3',
    'CX2_16': 'CALCE_CX2_T1', 'CX2_31': 'CALCE_CX2_T1',
    'CX2_33': 'CALCE_CX2_T1', 'CX2_35': 'CALCE_CX2_T1',
    'CX2_34': 'CALCE_CX2_T2', 'CX2_36': 'CALCE_CX2_T2',
    'CX2_37': 'CALCE_CX2_T2', 'CX2_38': 'CALCE_CX2_T2',
    'CX2_40': 'CALCE_CX2_T3', 'CX2_41': 'CALCE_CX2_T3',
    'CX2_42': 'CALCE_CX2_T3', 'CX2_43': 'CALCE_CX2_T3',
}
study_groups = np.array([study_group_map.get(b, 'UNKNOWN') for b in groups])
sg_uniq      = sorted(set(study_groups))
sg_to_int    = {sg: i for i, sg in enumerate(sg_uniq)}
domain_ids   = np.array([sg_to_int[sg] for sg in study_groups])
print('Study groups:', sg_uniq)

## Helpers

In [ ]:
def multi_seed_cv(model_factory, X, mask, y, groups, cidx, device,
                  splitter, n_seeds=N_SEEDS, base_seed=42, **kw):
    all_results = []
    for s in range(n_seeds):
        seed = base_seed + s * 7
        torch.manual_seed(seed)
        np.random.seed(seed)
        r = vgx.run_grouped_cv(
            model_factory, X, mask, y, groups, cidx, device,
            splitter, seed=seed, **kw
        )
        all_results.append(r)
    return all_results


def agg_seeds(all_results):
    aggs = [vg.aggregate(r) for r in all_results]
    out = {}
    for k in ['mae', 'rmse', 'r2', 'skill', 'spearman']:
        vals = [a[k] for a in aggs]
        out[k]               = float(np.mean(vals))
        out[k + '_seed_std'] = float(np.std(vals))
        out[k + '_std']      = float(np.mean([a.get(k + '_std', 0) for a in aggs]))
    return out


def fmt_seed(agg, key='mae'):
    return f"{agg[key]:.4f} ± {agg[key + '_seed_std']:.4f}"


def lodo_by_ds(all_results):
    by_ds = {}
    for results in all_results:
        for fold in results:
            ds = ','.join(fold.get('held_group') or [])
            by_ds.setdefault(ds, []).append(fold['metrics']['mae'])
    return {ds: {'mae': np.mean(v), 'mae_std': np.std(v)} for ds, v in by_ds.items()}


def params_to_aug(p):
    """Convert Optuna param dict (strength scalars) to a make_augment call + train_kwargs extras."""
    ws  = p.get('warp_strength',       0.0)
    tws = p.get('time_warp_strength',  0.0)
    aug_fn = src.vg_augment.make_augment(
        rate_warp_lo       = 1.0 - ws  if ws  > 0 else 1.0,
        rate_warp_hi       = 1.0 / (1.0 - ws)  if ws  > 0 else 1.0,
        jitter_sigma       = p.get('jitter_sigma', 0.0),
        time_warp_lo       = 1.0 - tws if tws > 0 else 1.0,
        time_warp_hi       = 1.0 / (1.0 - tws) if tws > 0 else 1.0,
        channel_dropout_p  = p.get('channel_dropout_p', 0.0),
    )
    train_kw = {
        'augment_fn':   aug_fn,
        'cutout_frac':  p.get('cutout_frac',  0.0),
        'gridshift_max': int(p.get('gridshift_max', 0)),
        'mixup_alpha':  p.get('mixup_alpha',  0.0),
        'use_amp':      USE_AMP,
    }
    return train_kw


def run_lodo_single(params, label=''):
    """Single-seed LODO pass. Returns (calce_mae, nasa_mae, by_ds)."""
    folds = vgx.run_grouped_cv(
        make_gru, X_base3, mask, y, groups, cidx, DEVICE,
        LeaveOneGroupOut(), cv_groups=ds_groups,
        epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
        scheduler=SCHEDULER, seed=42,
        progress=False, verbose=False,
        train_kwargs=params_to_aug(params),
    )
    by_ds    = lodo_by_ds([folds])
    calce    = by_ds.get('calce', {}).get('mae', 1.0)
    nasa     = by_ds.get('nasa',  {}).get('mae', 1.0)
    if label:
        print(f'{label}: CALCE={calce:.4f}  NASA={nasa:.4f}  mean={(calce+nasa)/2:.4f}')
    return calce, nasa, by_ds


def run_gkf_single(params, label=''):
    """Single-seed GKF-6 pass. Returns agg dict."""
    folds = vgx.run_grouped_cv(
        make_gru, X_base3, mask, y, groups, cidx, DEVICE,
        GroupKFold(n_splits=6), cv_groups=study_groups,
        epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
        scheduler=SCHEDULER, seed=42,
        progress=False, verbose=False,
        train_kwargs=params_to_aug(params),
    )
    agg = vg.aggregate(folds)
    if label:
        print(f'{label}: GKF-6 MAE={agg["mae"]:.4f}  R²={agg["r2"]:.3f}')
    return agg

## Reference model (nb05 best)

In [ ]:
make_gru = lambda: src.vg_models.VGGRUReg(n_features=3, hidden=47, dropout=0.35)

print('Reference (nb05 best Optuna aug) — LODO')
ref_calce, ref_nasa, ref_lodo_ds = run_lodo_single(NB05_BEST, label='  nb05_best')

print('Reference (nb05 best Optuna aug) — GKF-6')
ref_gkf = run_gkf_single(NB05_BEST, label='  nb05_best')

## New transforms — visual sanity check

Apply each new transform to a real sample and plot the leading valid block before/after
to confirm behaviour on the voltage-grid layout.

In [ ]:
from src.vg_extended import global_scale, _reapply_fill
from src.vg_augment import end_cutout, grid_shift, channel_dropout

# Grab a small batch from a real CALCE fold (already scaled)
_is_calce = ds_groups == 'calce'
_Xc = X_base3[_is_calce][:80]
_mc = mask[_is_calce][:80]
_Xc_s, _, _ = global_scale(_Xc, _Xc[:8], _Xc[:8], _mc)
_Xc_s = _reapply_fill(_Xc_s, _mc)

xb  = torch.tensor(_Xc_s[:8], dtype=torch.float32)
mb  = torch.tensor(_mc[:8],   dtype=torch.bool)
sample_i = 0  # which sample to plot
L = xb.shape[1]
grid = np.arange(L)

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharey=False)
fig.suptitle('New augmentation transforms on voltage-grid data (sample 0)', fontsize=13)
ch_labels = ['C-rate', 'ΔT', 't_elapsed']

for ch_i, ch_name in enumerate(ch_labels):
    orig_vals  = xb[sample_i, :, ch_i].numpy()
    orig_mask  = mb[sample_i].numpy()
    valid_len  = int(orig_mask.sum())

    # cutout
    xb_cut, mb_cut = end_cutout(xb, mb, frac_max=0.30)
    cut_vals = xb_cut[sample_i, :, ch_i].numpy()
    cut_mask = mb_cut[sample_i].numpy()

    # grid_shift (k=+5)
    xb_sh = xb.clone(); mb_sh = mb.clone()
    # apply deterministic shift for visualization
    import src.vg_augment as _va
    xb_sh2, mb_sh2 = _va.grid_shift(xb, mb, max_shift=5)
    sh_vals  = xb_sh2[sample_i, :, ch_i].numpy()
    sh_mask  = mb_sh2[sample_i].numpy()

    ax_orig = axes[ch_i, 0]
    ax_cut  = axes[ch_i, 1]
    ax_sh   = axes[ch_i, 2]
    ax_mix  = axes[ch_i, 3]

    ax_orig.plot(grid[orig_mask], orig_vals[orig_mask], 'C0', lw=1.5, label='original')
    ax_orig.set_title(f'{ch_name} — original (len={valid_len})')

    ax_cut.plot(grid[orig_mask], orig_vals[orig_mask], 'C0', alpha=0.3, lw=1, label='original')
    ax_cut.plot(grid[cut_mask], cut_vals[cut_mask], 'C1', lw=1.5, label='after cutout')
    ax_cut.set_title(f'{ch_name} — end_cutout(frac≤0.30)')

    ax_sh.plot(grid[orig_mask], orig_vals[orig_mask], 'C0', alpha=0.3, lw=1, label='original')
    ax_sh.plot(grid[sh_mask], sh_vals[sh_mask], 'C2', lw=1.5, label='after shift')
    ax_sh.set_title(f'{ch_name} — grid_shift(max=5)')

    # mixup: show λ=0.6 blend with a random partner
    lam = 0.6
    partner_i = (sample_i + 3) % 8
    mix_vals  = lam * xb[sample_i, :, ch_i].numpy() + (1-lam) * xb[partner_i, :, ch_i].numpy()
    mix_mask  = (mb[sample_i] & mb[partner_i]).numpy()
    ax_mix.plot(grid[orig_mask], orig_vals[orig_mask], 'C0', alpha=0.3, lw=1, label='original')
    ax_mix.plot(grid[mix_mask], mix_vals[mix_mask], 'C3', lw=1.5, label=f'mixup λ={lam}')
    ax_mix.set_title(f'{ch_name} — mixup(λ={lam})')

for ax in axes.flat:
    ax.set_xlabel('voltage grid index →')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(RESULTS / '06_transform_sanity.png', dpi=120)
plt.show()

## Experiment A — Single-axis ablations

Each new axis is added one at a time on top of the nb05 best, to attribute independent
gains before the joint sweep.  All runs: single-seed, LODO only (GKF-6 added for winners).

In [ ]:
from tqdm.auto import tqdm

# Each entry: (label, extra_params_over_nb05_best)
ABLATIONS = [
    ('nb05_best (baseline)',       {}),
    ('+ cutout_frac=0.15',         {'cutout_frac': 0.15}),
    ('+ cutout_frac=0.25',         {'cutout_frac': 0.25}),
    ('+ gridshift_max=3',          {'gridshift_max': 3}),
    ('+ gridshift_max=6',          {'gridshift_max': 6}),
    ('+ mixup_alpha=0.3',          {'mixup_alpha': 0.3}),
    ('+ mixup_alpha=0.5',          {'mixup_alpha': 0.5}),
    ('+ channel_dropout_p=0.1',    {'channel_dropout_p': 0.1}),
    ('+ channel_dropout_p=0.2',    {'channel_dropout_p': 0.2}),
]

ablation_rows = []
for label, extra in tqdm(ABLATIONS, desc='ablations'):
    params = {**NB05_BEST, **extra}
    calce, nasa, _ = run_lodo_single(params, label=f'  {label}')
    ablation_rows.append({'label': label, 'CALCE': calce, 'NASA': nasa, 'mean': (calce+nasa)/2})

abl_df = pd.DataFrame(ablation_rows).set_index('label')
abl_df['Δ_mean'] = abl_df['mean'] - abl_df['mean'].iloc[0]
display(abl_df.round(4))

### Conclusion A

*(Fill in after running — note which axes help/hurt CALCE vs NASA independently.)*

## Experiment B — Multi-objective Optuna sweep (NSGA-II)

Search space (7-D):

| Parameter | Range | Type |
|---|---|---|
| `warp_strength` | [0, 0.50] | float |
| `jitter_sigma` | [0, 0.02] | float |
| `time_warp_strength` | [0, 0.30] | float |
| `cutout_frac` | [0, 0.30] | float |
| `gridshift_max` | [0, 8] | int |
| `mixup_alpha` | [0, 0.50] | float |
| `channel_dropout_p` | [0, 0.30] | float |

Objectives: minimise **(CALCE MAE, NASA MAE)** simultaneously.
Sampler: NSGA-II (population_size=25, ~4 generations over 100 trials).

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

RERUN     = False   # set True to re-run the sweep (~1 h)
STUDY_PKL = RESULTS / '06_aug_study.pkl'

# Best known params from sweep (update after running)
BALANCED_PARAMS = None  # will be set from study.best_trials after run


def _aug_objective(trial):
    warp_s    = trial.suggest_float('warp_strength',       0.0, 0.50)
    jitter    = trial.suggest_float('jitter_sigma',        0.0, 0.02)
    twarp_s   = trial.suggest_float('time_warp_strength',  0.0, 0.30)
    cutout    = trial.suggest_float('cutout_frac',         0.0, 0.30)
    gridshift = trial.suggest_int(  'gridshift_max',       0,   8)
    mixup     = trial.suggest_float('mixup_alpha',         0.0, 0.50)
    ch_drop   = trial.suggest_float('channel_dropout_p',   0.0, 0.30)

    params = {
        'warp_strength':      warp_s,
        'jitter_sigma':       jitter,
        'time_warp_strength': twarp_s,
        'cutout_frac':        cutout,
        'gridshift_max':      gridshift,
        'mixup_alpha':        mixup,
        'channel_dropout_p':  ch_drop,
    }
    folds = vgx.run_grouped_cv(
        make_gru, X_base3, mask, y, groups, cidx, DEVICE,
        LeaveOneGroupOut(), cv_groups=ds_groups,
        epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
        scheduler=SCHEDULER, seed=42,
        progress=False, verbose=False,
        train_kwargs=params_to_aug(params),
    )
    by_ds     = lodo_by_ds([folds])
    calce_mae = by_ds.get('calce', {}).get('mae', 1.0)
    nasa_mae  = by_ds.get('nasa',  {}).get('mae', 1.0)

    n_pareto  = len(aug_study.best_trials) if len(aug_study.trials) > 1 else 0
    tqdm.write(
        f't{trial.number:3d}  ws={warp_s:.3f}  jit={jitter:.4f}  tw={twarp_s:.3f}'
        f'  cut={cutout:.3f}  gs={gridshift}  mix={mixup:.3f}  ch={ch_drop:.3f}'
        f'  →  calce={calce_mae:.4f}  nasa={nasa_mae:.4f}  pareto={n_pareto}'
    )
    return calce_mae, nasa_mae


if RERUN:
    sampler   = optuna.samplers.NSGAIISampler(population_size=25, seed=42)
    aug_study = optuna.create_study(
        directions=['minimize', 'minimize'],
        sampler=sampler,
        study_name='aug_sweep_mo',
    )

    # Enqueued anchor trials (run before NSGA-II takes over)
    _anchors = [
        {'warp_strength': 0.0,   'jitter_sigma': 0.0,    'time_warp_strength': 0.0,
         'cutout_frac': 0.0,    'gridshift_max': 0,     'mixup_alpha': 0.0,   'channel_dropout_p': 0.0},
        {'warp_strength': 0.2,   'jitter_sigma': 0.01,   'time_warp_strength': 0.0,
         'cutout_frac': 0.0,    'gridshift_max': 0,     'mixup_alpha': 0.0,   'channel_dropout_p': 0.0},
        {'warp_strength': 0.07,  'jitter_sigma': 0.006,  'time_warp_strength': 0.11,
         'cutout_frac': 0.0,    'gridshift_max': 0,     'mixup_alpha': 0.0,   'channel_dropout_p': 0.0},
        # single-axis probes on top of nb05 best
        {'warp_strength': 0.07,  'jitter_sigma': 0.006,  'time_warp_strength': 0.11,
         'cutout_frac': 0.15,   'gridshift_max': 0,     'mixup_alpha': 0.0,   'channel_dropout_p': 0.0},
        {'warp_strength': 0.07,  'jitter_sigma': 0.006,  'time_warp_strength': 0.11,
         'cutout_frac': 0.0,    'gridshift_max': 3,     'mixup_alpha': 0.0,   'channel_dropout_p': 0.0},
        {'warp_strength': 0.07,  'jitter_sigma': 0.006,  'time_warp_strength': 0.11,
         'cutout_frac': 0.0,    'gridshift_max': 0,     'mixup_alpha': 0.3,   'channel_dropout_p': 0.0},
        {'warp_strength': 0.07,  'jitter_sigma': 0.006,  'time_warp_strength': 0.11,
         'cutout_frac': 0.0,    'gridshift_max': 0,     'mixup_alpha': 0.0,   'channel_dropout_p': 0.1},
    ]
    for a in _anchors:
        aug_study.enqueue_trial(a)

    tqdm.write(f'Running NSGA-II multi-objective sweep: {N_TRIALS} trials')
    aug_study.optimize(_aug_objective, n_trials=N_TRIALS, show_progress_bar=False)

    with open(STUDY_PKL, 'wb') as f:
        pickle.dump(aug_study, f)
    tqdm.write(f'Study saved → {STUDY_PKL}')
else:
    tqdm.write('Loading cached study...')
    with open(STUDY_PKL, 'rb') as f:
        aug_study = pickle.load(f)
    tqdm.write(f'Loaded {len(aug_study.trials)} trials, {len(aug_study.best_trials)} Pareto-optimal')

In [ ]:
# Optuna built-in Pareto front
from optuna.visualization import plot_pareto_front, plot_param_importances

fig_pareto = plot_pareto_front(aug_study, target_names=['CALCE MAE', 'NASA MAE'])
fig_pareto.show()

In [ ]:
# Custom Plotly scatter — all trials with full param hover, Pareto highlighted
all_trials = [t for t in aug_study.trials if t.values is not None]
pareto_nums = {t.number for t in aug_study.best_trials}

PARAM_NAMES = [
    'warp_strength', 'jitter_sigma', 'time_warp_strength',
    'cutout_frac', 'gridshift_max', 'mixup_alpha', 'channel_dropout_p',
]

def _hover(t):
    p = t.params
    lines = [
        f'trial #{t.number}',
        f'CALCE={t.values[0]:.4f}  NASA={t.values[1]:.4f}',
        '—',
    ] + [f'{k}={p.get(k, "?"):.4g}' if k != 'gridshift_max'
         else f'{k}={int(p.get(k, 0))}'
         for k in PARAM_NAMES]
    return '<br>'.join(lines)

calce_all  = [t.values[0]  for t in all_trials]
nasa_all   = [t.values[1]  for t in all_trials]
hover_all  = [_hover(t)    for t in all_trials]
colors     = ['red' if t.number in pareto_nums else 'steelblue' for t in all_trials]
sizes      = [12   if t.number in pareto_nums else 6            for t in all_trials]

fig_custom = go.Figure()
fig_custom.add_trace(go.Scatter(
    x=calce_all, y=nasa_all,
    mode='markers',
    marker=dict(color=colors, size=sizes, opacity=0.75, line=dict(width=0.5, color='white')),
    hovertemplate='%{customdata}<extra></extra>',
    customdata=hover_all,
    name='all trials',
))

# Mark the nb05 reference
fig_custom.add_trace(go.Scatter(
    x=[ref_calce], y=[ref_nasa],
    mode='markers+text',
    marker=dict(symbol='diamond', color='gold', size=14, line=dict(width=1, color='black')),
    text=['nb05 best'], textposition='top right',
    name='nb05 best',
))

fig_custom.update_layout(
    title='Multi-objective sweep — all trials (red = Pareto-optimal)',
    xaxis_title='CALCE MAE (lower is better →)',
    yaxis_title='NASA MAE (lower is better →)',
    width=750, height=550,
)
fig_custom.show()
fig_custom.write_html(str(RESULTS / '06_pareto_custom.html'))

In [ ]:
# Per-objective parameter importances
for obj_i, obj_name in enumerate(['CALCE MAE', 'NASA MAE']):
    fig_imp = plot_param_importances(
        aug_study,
        target=lambda t, i=obj_i: t.values[i],
        target_name=obj_name,
    )
    fig_imp.update_layout(title=f'Param importance → {obj_name}')
    fig_imp.show()

### Conclusion B

*(Fill in after running — identify dominant axes per objective, note Pareto structure.)*

## Experiment C — Operating point validation

Pick ~3 representative points from the Pareto front and confirm on full LODO + GKF-6:
- **Balanced**: minimise CALCE+NASA sum (similar to the nb05 single-objective)
- **NASA-leaning**: minimise NASA MAE
- **CALCE-leaning**: minimise CALCE MAE

In [ ]:
pareto = aug_study.best_trials

balanced_t  = min(pareto, key=lambda t: t.values[0] + t.values[1])
nasa_best_t = min(pareto, key=lambda t: t.values[1])
calce_best_t= min(pareto, key=lambda t: t.values[0])

operating_points = [
    ('balanced',      balanced_t),
    ('nasa_leaning',  nasa_best_t),
    ('calce_leaning', calce_best_t),
]

print('Selected operating points from Pareto front:')
for name, t in operating_points:
    print(f'  {name}: trial #{t.number}  CALCE={t.values[0]:.4f}  NASA={t.values[1]:.4f}')
    print(f'    params: {t.params}')

In [ ]:
op_results = {}
for name, t in operating_points:
    print(f'\n--- {name} (trial #{t.number}) ---')
    calce, nasa, _ = run_lodo_single(t.params, label=f'  LODO {name}')
    gkf_agg = run_gkf_single(t.params, label=f'  GKF-6 {name}')
    op_results[name] = {
        'params': t.params,
        'lodo_calce': calce, 'lodo_nasa': nasa, 'lodo_mean': (calce+nasa)/2,
        'gkf_mae': gkf_agg['mae'], 'gkf_r2': gkf_agg['r2'],
    }

op_df = pd.DataFrame(op_results).T
print('\nOperating point summary:')
display(op_df[['lodo_calce','lodo_nasa','lodo_mean','gkf_mae','gkf_r2']].round(4))

### Conclusion C

*(Fill in: which point to carry forward as the new best? Note tradeoffs.)*

## Experiment D — C2 1-shot re-check

Run the nb05 C2 protocol (fine-tune fc head on k=1 target cell) on the new best base model
to see if a better zero-shot model shifts the few-shot ceiling.

In [ ]:
from src.vg_extended import global_scale, _reapply_fill, lobo_metrics
from src.vg_models import predict_vg
from src import sequence

# Choose the balanced operating point as the new best (update if conclusion C suggests otherwise)
CHOSEN_PARAMS = op_results['balanced']['params']
print(f'Chosen params: {CHOSEN_PARAMS}')


def run_lodo_tta(model_factory, X, mask, y, groups, cidx, device, cv_groups, params):
    """LODO loop that retains trained models for C2 adaptation."""
    splitter  = LeaveOneGroupOut()
    train_kw  = params_to_aug(params)
    folds_out = []

    for fold_idx, (_, test_idx) in enumerate(splitter.split(X, groups=cv_groups)):
        is_test  = np.zeros(len(X), dtype=bool)
        is_test[test_idx] = True
        is_avail = ~is_test

        avail_bids = sorted(set(groups[is_avail]))
        bid_sg     = {b: cv_groups[is_avail & (groups == b)][0] for b in avail_bids}
        sg_counts  = {}
        for b in avail_bids:
            sg = bid_sg[b]
            sg_counts[sg] = sg_counts.get(sg, 0) + int((is_avail & (groups == b)).sum())
        largest_sg = max(sg_counts, key=lambda k: sg_counts[k])
        val_bid    = sorted(b for b, sg in bid_sg.items() if sg == largest_sg)[-1]
        is_val     = is_avail & (groups == val_bid)
        is_train   = is_avail & ~is_val

        X_tr_s, X_val_s, X_te_s = global_scale(X[is_train], X[is_val], X[is_test], mask[is_train])
        X_tr_s  = _reapply_fill(X_tr_s,  mask[is_train])
        X_val_s = _reapply_fill(X_val_s, mask[is_val])
        X_te_s  = _reapply_fill(X_te_s,  mask[is_test])

        torch.manual_seed(fold_idx * 31 + 42)
        model = model_factory().to(device)
        model, _, _, best_ep = sequence.train_model(
            model, X_tr_s, mask[is_train], y[is_train],
            X_val_s, mask[is_val], y[is_val],
            device, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
            patience=PATIENCE, scheduler=SCHEDULER, **train_kw,
        )
        model.eval()

        y_pred_base = predict_vg(model, X_te_s, mask[is_test], device)
        base_m      = lobo_metrics(y[is_test], y_pred_base, cidx[is_test], y[is_train].mean())

        held_str = ','.join(sorted(set(cv_groups[is_test])))
        print(f'  [f={fold_idx}] held={held_str}  base MAE={base_m["mae"]:.4f}  ep={best_ep}')

        folds_out.append(dict(
            fold_idx=fold_idx,
            held_group=sorted(set(cv_groups[is_test])),
            model=model,
            X_te_s=X_te_s, mask_te=mask[is_test],
            y_te=y[is_test], groups_te=groups[is_test],
            cidx_te=cidx[is_test],
            y_tr_mean=y[is_train].mean(),
            baseline_metrics=base_m,
        ))
    return folds_out


print('Training LODO models for C2 adaptation...')
tta_folds = run_lodo_tta(
    make_gru, X_base3, mask, y, groups, cidx, DEVICE,
    cv_groups=ds_groups, params=CHOSEN_PARAMS,
)

In [ ]:
def few_shot_adapt(model, X_te_s, mask_te, y_te, groups_te, cidx_te, y_tr_mean,
                   device, k_cells=1, n_epochs=30, ft_lr=5e-4):
    all_cells = sorted(set(groups_te))
    ft_cells  = all_cells[:k_cells]
    ev_cells  = all_cells[k_cells:]
    if not ev_cells:
        return None

    is_ft = np.isin(groups_te, ft_cells)
    is_ev = np.isin(groups_te, ev_cells)

    Xev_t = torch.tensor(X_te_s[is_ev], dtype=torch.float32, device=device)
    mev_t = torch.tensor(mask_te[is_ev], dtype=torch.bool,    device=device)

    model.eval()
    with torch.no_grad():
        pred_base = model(Xev_t, mev_t).cpu().numpy()
    base_m = lobo_metrics(y_te[is_ev], pred_base, cidx_te[is_ev], y_tr_mean)

    model_ft = copy.deepcopy(model)
    for name, param in model_ft.named_parameters():
        param.requires_grad = 'fc' in name

    opt  = torch.optim.Adam([p for p in model_ft.parameters() if p.requires_grad], lr=ft_lr)
    l1   = nn.L1Loss()
    Xft  = torch.tensor(X_te_s[is_ft], dtype=torch.float32, device=device)
    mft  = torch.tensor(mask_te[is_ft], dtype=torch.bool,    device=device)
    yft  = torch.tensor(y_te[is_ft],   dtype=torch.float32, device=device)

    model_ft.train()
    for _ in range(n_epochs):
        opt.zero_grad()
        l1(model_ft(Xft, mft), yft).backward()
        opt.step()

    model_ft.eval()
    with torch.no_grad():
        pred_ev = model_ft(Xev_t, mev_t).cpu().numpy()
    adapt_m = lobo_metrics(y_te[is_ev], pred_ev, cidx_te[is_ev], y_tr_mean)

    return {
        'adapted_mae': adapt_m['mae'],
        'noadapt_mae': base_m['mae'],
        'delta':       adapt_m['mae'] - base_m['mae'],
    }


K_SHOTS = [1, 3, 5]
c2_results = {k: {} for k in K_SHOTS}

for fd in tta_folds:
    ds = ','.join(fd['held_group'])
    for k in K_SHOTS:
        m = few_shot_adapt(
            fd['model'], fd['X_te_s'], fd['mask_te'],
            fd['y_te'], fd['groups_te'], fd['cidx_te'],
            fd['y_tr_mean'], DEVICE, k_cells=k,
        )
        if m is not None:
            c2_results[k].setdefault(ds, []).append(m)

print('C2 — few-shot head fine-tune (delta = adapted − no-adapt; negative = improvement)')
for k in K_SHOTS:
    print(f'  k={k} cells:')
    for ds, entries in sorted(c2_results[k].items()):
        adapted = np.mean([e['adapted_mae'] for e in entries])
        noadapt = np.mean([e['noadapt_mae'] for e in entries])
        delta   = np.mean([e['delta']       for e in entries])
        print(f'    held={ds}: no-adapt={noadapt:.4f}  adapted={adapted:.4f}  Δ={delta:+.4f}')

### Conclusion D

*(Fill in: does the new base model improve the few-shot ceiling vs nb05? Is k=1 still optimal?)*

## Final presentation metrics

Chosen winner vs nb05 best — LODO + GKF-6.  Single-seed (seed-std ≈ 0 from nb05 experience;
run N_SEEDS > 1 here only if the presentation requires it).

In [ ]:
FINAL_N_SEEDS = 3  # bump to 5 if time allows

print('Final metrics — nb05 best (LODO)')
nb05_lodo_seeds = multi_seed_cv(
    make_gru, X_base3, mask, y, groups, cidx, DEVICE,
    LeaveOneGroupOut(), cv_groups=ds_groups, n_seeds=FINAL_N_SEEDS,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
    train_kwargs=params_to_aug(NB05_BEST),
)
nb05_lodo_agg = agg_seeds(nb05_lodo_seeds)
nb05_lodo_ds  = lodo_by_ds(nb05_lodo_seeds)
print(f'  LODO MAE {fmt_seed(nb05_lodo_agg)}')
for ds, v in sorted(nb05_lodo_ds.items()):
    print(f'    held={ds}: MAE {v["mae"]:.4f} ± {v["mae_std"]:.4f}')

print('\nFinal metrics — nb06 best (LODO)')
nb06_lodo_seeds = multi_seed_cv(
    make_gru, X_base3, mask, y, groups, cidx, DEVICE,
    LeaveOneGroupOut(), cv_groups=ds_groups, n_seeds=FINAL_N_SEEDS,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
    train_kwargs=params_to_aug(CHOSEN_PARAMS),
)
nb06_lodo_agg = agg_seeds(nb06_lodo_seeds)
nb06_lodo_ds  = lodo_by_ds(nb06_lodo_seeds)
print(f'  LODO MAE {fmt_seed(nb06_lodo_agg)}')
for ds, v in sorted(nb06_lodo_ds.items()):
    print(f'    held={ds}: MAE {v["mae"]:.4f} ± {v["mae_std"]:.4f}')

In [ ]:
print('Final metrics — nb05 best (GKF-6)')
nb05_gkf_seeds = multi_seed_cv(
    make_gru, X_base3, mask, y, groups, cidx, DEVICE,
    GroupKFold(n_splits=6), cv_groups=study_groups, n_seeds=FINAL_N_SEEDS,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
    train_kwargs=params_to_aug(NB05_BEST),
)
nb05_gkf_agg = agg_seeds(nb05_gkf_seeds)
print(f'  GKF-6 MAE {fmt_seed(nb05_gkf_agg)}  R² {fmt_seed(nb05_gkf_agg, "r2")}')

print('\nFinal metrics — nb06 best (GKF-6)')
nb06_gkf_seeds = multi_seed_cv(
    make_gru, X_base3, mask, y, groups, cidx, DEVICE,
    GroupKFold(n_splits=6), cv_groups=study_groups, n_seeds=FINAL_N_SEEDS,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    scheduler=SCHEDULER,
    train_kwargs=params_to_aug(CHOSEN_PARAMS),
)
nb06_gkf_agg = agg_seeds(nb06_gkf_seeds)
print(f'  GKF-6 MAE {fmt_seed(nb06_gkf_agg)}  R² {fmt_seed(nb06_gkf_agg, "r2")}')

In [ ]:
# Write summary CSVs
lodo_rows = []
for label, agg, ds_agg in [
    ('nb05_best', nb05_lodo_agg, lodo_by_ds(nb05_lodo_seeds)),
    ('nb06_best', nb06_lodo_agg, lodo_by_ds(nb06_lodo_seeds)),
]:
    row = {'model': label, **{k: agg[k] for k in ['mae','mae_seed_std','rmse','r2']}}
    for ds, v in ds_agg.items():
        row[f'lodo_{ds}_mae'] = v['mae']
    lodo_rows.append(row)

gkf_rows = []
for label, agg in [
    ('nb05_best', nb05_gkf_agg),
    ('nb06_best', nb06_gkf_agg),
]:
    gkf_rows.append({'model': label, **{k: agg[k] for k in ['mae','mae_seed_std','rmse','r2']}})

lodo_df = pd.DataFrame(lodo_rows).set_index('model')
gkf_df  = pd.DataFrame(gkf_rows).set_index('model')

lodo_df.to_csv(RESULTS / '06_lodo_summary.csv')
gkf_df.to_csv(RESULTS  / '06_gkf_summary.csv')
print('Saved 06_lodo_summary.csv and 06_gkf_summary.csv')

print('\nLODO summary:')
display(lodo_df.round(4))
print('\nGKF-6 summary:')
display(gkf_df.round(4))

## Overall conclusions

### Results

*(Update after running — copy nb05 table and extend with nb06 rows)*

| Method | LODO mean | LODO CALCE | LODO NASA | GKF-6 |
|---|---|---|---|---|
| Anchor (nb04) | 0.0645 | 0.0587 | 0.0703 | 0.0624 |
| **Optuna aug (nb05)** | 0.0580 | 0.0514 | 0.0646 | 0.0545 |
| nb05 + C2 1-shot (k=1) | — | 0.0681 | **0.0452** | — |
| **nb06 balanced** | — | — | — | — |
| nb06 + C2 1-shot (k=1) | — | — | — | — |

### Key findings

*(Fill in after running)*

### Augmentation axes — final ranking

*(Which axes added independent value? Which were dominated?)*

### CALCE/NASA Pareto tradeoff

*(Was there a Pareto-dominant point, or a genuine tradeoff? What does the front shape say about the datasets?)*